In [ ]:
import pandas as pd
from pathlib import Path


In [ ]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/Data Preparation and Analysis")

In [ ]:
# ...existing code...
from sklearn.model_selection import train_test_split

train_logs = pd.read_csv(data_path / "train_logs.csv")

# podział 80/20
X_train_scores, X_test_scores = train_test_split(
    train_logs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# opcjonalnie: zapis, aby w kolejnych uruchomieniach móc je wczytywać z plików
X_train_scores.to_csv(data_path / "X_train_scores.csv", index=False)
X_test_scores.to_csv(data_path / "X_test_scores.csv", index=False)

print("Train:", X_train_scores.shape, "Test:", X_test_scores.shape)
# ...existing code...

In [ ]:
train_logs.head(10)

In [ ]:
chars_per_essay = train_logs[train_logs['activity'] == 'Input'].groupby('id').agg(chars_per_essay=('event_id','nunique'))

In [ ]:
chars_per_essay

In [ ]:
grouped_train_logs = train_logs.groupby("id").agg(
    words_per_essay=("word_count", "last"),
    max_up_time=("up_time", "max"),
    min_down_time=("down_time", "min"),
    event_per_essay=("event_id", "nunique")
)

In [ ]:
grouped_train_logs

In [ ]:
grouped_train_logs ['writing_duration'] = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

In [ ]:
grouped_train_logs = grouped_train_logs.merge(chars_per_essay, left_index=True, right_index=True)

In [ ]:
grouped_train_logs = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

In [ ]:
grouped_train_logs ['chars_per_minute'] = 1000 * 60  grouped_train_logs['chars_per_essay'] / grouped_train_logs['wrtiting_duration']

### Pauses

In [ ]:
train_logs.sort_values(by=['id', 'down_time'], inplace=True)
train_logs['pause_time'] = (train_logs.groupby('id')['down_time'].shift(0) - train_logs.groupby('id')['up_time'].shift(1)).fillna(0)

In [ ]:
train_logs

In [ ]:
train_logs['is pause'] = train_logs['pause_time'] > 2000

In [ ]:
pause_features = train_logs.groupby('id').agg(
    total_pauses=('pause_time', 'sum'),
    total_pause_time=('pause_time', 'sum')
)

In [ ]:
pause_features

In [ ]:
grouped_train_logs = grouped_train_logs.merge(pause_features, left_index=True, right_index=True)

In [ ]:
grouped_train_logs

In [ ]:
grouped_train_logs['perc_pause_time'] = grouped_train_logs['total_pause_time'] / grouped_train_logs['writing_duration']

In [ ]:
grouped_train_logs['pauses_per_words'] = grouped_train_logs['total_pauses'] / grouped_train_logs['words_per_essay']